In [1]:
# IMPORTS

from __future__ import annotations
from minigrid.core.constants import COLOR_NAMES
from minigrid.core.grid import Grid
from minigrid.core.mission import MissionSpace
from minigrid.core.world_object import Door, Goal, Key, Wall
from minigrid.minigrid_env import MiniGridEnv
from minigrid.envs import EmptyEnv
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

C:\Users\Fcomm\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#CREATE ENVIRONMENT
class SimpleEnv(MiniGridEnv):
    def __init__(self, map_type=0, size=7, agent_start_pos=(1,1), agent_start_dir=0, max_steps=None, **kwargs):
        self.map_type = map_type
        self.agent_start_pos = agent_start_pos
        self.agent_start_dir = agent_start_dir

        mission_space = MissionSpace(mission_func=self._gen_mission)

        if max_steps is None:
            max_steps = 4 * size**2

        super().__init__(
            mission_space=mission_space,
            grid_size=size,
            see_through_walls=True,
            max_steps=max_steps,
            **kwargs,
    )
    
    @staticmethod
    def _gen_mission():
        return "grand mission"

    def _gen_grid(self, width, height):

        self.grid = Grid(width, height)
        self.grid.wall_rect(0, 0, width, height)

        if self.map_type == 0:
            # simple open map
            pass

        elif self.map_type == 1:
            # corridor
            for y in range(1, height-1):
                self.grid.set(3, y, Wall())

        elif self.map_type == 2:
            # small maze block
            self.grid.wall_rect(2, 2, 3, 3)

        elif self.map_type == 3:
            # dead end test
            self.grid.set(2, 1, Wall())
            self.grid.set(2, 2, Wall())
            self.grid.set(2, 3, Wall())

        # goal always placed
        self.put_obj(Goal(), width - 2, height - 2)

        self.agent_pos = self.agent_start_pos
        self.agent_dir = self.agent_start_dir
        
        self.mission = "grand mission"

In [3]:
#LLM caller
class LLM:

    def __init__(self):

        model_name = "google/flan-t5-small"

        print("Loading tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        print("Loading model...")
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


    def ask(self, prompt):

        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=256
        )


        outputs = self.model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False
        )
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        return response

llm = LLM()

Loading tokenizer...


Loading model...


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 1120.01it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [4]:
def decision_to_action(text):
    text = text.lower()

    if "left" in text:
        return 0
    elif "right" in text:
        return 1
    elif "front" in text or "forward" in text:
        return 2
    else:
        return 3  # skip

In [5]:
def obs_to_state(obs):
    grid = obs["image"]

    def decode(tile):
        obj = tile[0]
        if obj == 2:
            return "wall"
        elif obj == 8:
            return "goal"
        else:
            return "floor"

    return {
        "Left": decode(grid[2][3]),
        "Front": decode(grid[3][4]),
        "Right": decode(grid[4][3])
    }

In [6]:
def local_agent(view_name, info):
    prompt = f"""
Environment:
- {info[0][0]}: {info[0][1]}
- {info[1][0]}: {info[1][1]}

Question:
Which direction is blocked?

Answer ONLY one word:
{info[0][0]} or {info[1][0]}.
Do not explain.
"""
    return llm.ask(prompt)

In [7]:
def get_views(state_dict):
    return {
        "left_front": [("Left", state_dict["Left"]), ("Front", state_dict["Front"])],
        "front_right": [("Front", state_dict["Front"]), ("Right", state_dict["Right"])],
        "left_right": [("Left", state_dict["Left"]), ("Right", state_dict["Right"])]
    }

In [8]:
def run_local_agents(state_dict):
    views = get_views(state_dict)
    outputs = {}

    for name, info in views.items():
        outputs[name] = local_agent(name, info)

    return outputs

In [9]:
def main_agent(state_dict, local_outputs):
    prompt = f"""
Environment:
- Left: {state_dict["Left"]}
- Front: {state_dict["Front"]}
- Right: {state_dict["Right"]}

Local observations:
- Left/Front agent says blocked: {local_outputs["left_front"]}
- Front/Right agent says blocked: {local_outputs["front_right"]}
- Left/Right agent says blocked: {local_outputs["left_right"]}

Question: Which direction should the agent move?

Rules:
- Do not choose a blocked direction
- Prefer open paths

Answer with one word: Left, Front, or Right.
"""
    return llm.ask(prompt)

In [10]:
def step(state_dict):
    local_outputs = run_local_agents(state_dict)

    print("Local outputs:", local_outputs)

    decision = main_agent(state_dict, local_outputs)

    return decision

In [11]:
def main():

    env = SimpleEnv(render_mode="human")
    obs, info = env.reset()

    done = False
    limitor = 0

    while not done:

        state = obs_to_state(obs)

        decision = step(state)

        print("FINAL:", decision)

        action = decision_to_action(decision)

        if action == 3:
            limitor += 1
            continue

        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        limitor += 1
        if limitor == 10:
            break

    env.close()

In [12]:
main()

Local outputs: {'left_front': 'Front', 'front_right': 'Front', 'left_right': 'Right'}
FINAL: Left
Local outputs: {'left_front': 'Front', 'front_right': 'Front', 'left_right': 'Right'}
FINAL: Left
Local outputs: {'left_front': 'Front', 'front_right': 'Front', 'left_right': 'Right'}
FINAL: Left
Local outputs: {'left_front': 'Front', 'front_right': 'Front', 'left_right': 'Right'}
FINAL: Left
Local outputs: {'left_front': 'Front', 'front_right': 'Front', 'left_right': 'Right'}
FINAL: Left
Local outputs: {'left_front': 'Front', 'front_right': 'Front', 'left_right': 'Right'}
FINAL: Left
Local outputs: {'left_front': 'Front', 'front_right': 'Front', 'left_right': 'Right'}
FINAL: Left
Local outputs: {'left_front': 'Front', 'front_right': 'Front', 'left_right': 'Right'}
FINAL: Left
Local outputs: {'left_front': 'Front', 'front_right': 'Front', 'left_right': 'Right'}
FINAL: Left
Local outputs: {'left_front': 'Front', 'front_right': 'Front', 'left_right': 'Right'}
FINAL: Left
